# The co-training dial + substrate climb — aleph adapter on a linear trunk

**The question.** Snap an aleph adapter onto a miniature 4-block linear
MNIST classifier and train both at once. What happens?

That is not one run. The campaign record already answers the two extremes,
and they disagree:

| regime | finding | evidence |
|---|---|---|
| **co-trained** | the address-bottleneck prior **pays** | exp012: beat an unrestricted head 7/7 across seeds and budgets; held at d=384 |
| **frozen substrate** | it **costs** | exp013 Track A: param-matched MLP wins, aleph tax ~ +0.09 CE, 2 seeds |

Both are canon. **Nothing between them has ever been measured.** The plan
that would have measured it was ranked first in the standing queue and got
displaced before it ran.

So we turn a dial instead of firing one run:

```
trainable trunk blocks:   0        1        2        4
                       frozen  --------------->  full co-training
                    (exp013 pole)             (exp012 pole)
```

Where the curve crosses zero — if it does — is the boundary law.

**Preregistered forks** (write the verdict from whichever fires; all are
canon-grade either way) are listed in
[`experiments/README.md`](../README.md). Read them before you look at a
number, not after.

Runtime: ~3 min quick pass, ~25 min full sweep. Runs on CPU; a T4 is plenty.

## 0 · Setup — one cell, nothing preinstalled

This is the whole dependency story. In a **fresh Colab runtime** it clones
the `experimental` branch (where `experiments/` lives — it is **not** on
`main`), puts `src/` and `experiments/` on `sys.path`, and imports.

No editable `pip install` and no build step: `amoe`'s only hard dependency
is `torch`, which Colab already ships, so pointing `sys.path` at the
checkout is enough. `torchvision` (also preinstalled on Colab) is fetched
only if genuinely missing. If you already have a local `experimental`
checkout, it uses that and clones nothing.

Idempotent: safe to re-run, safe to paste into a fresh runtime.

In [ ]:
import os, sys, subprocess

REPO   = 'https://github.com/AbstractEyes/amoe-lora'
BRANCH = 'experimental'          # experiments/ lives here, not on main
IN_COLAB = 'google.colab' in sys.modules

def find_root(start='.'):
    """Walk up for a checkout that actually has the experiment code.
    Requires experiments/ so a bare `main` checkout won't match. No
    __file__ reliance — this cell must survive being pasted anywhere."""
    p = os.path.abspath(start)
    while p != os.path.dirname(p):
        if (os.path.isfile(os.path.join(p, 'pyproject.toml')) and
                os.path.isdir(os.path.join(p, 'experiments', 'aleph_mnist'))):
            return p
        p = os.path.dirname(p)
    return None

ROOT = find_root()
if ROOT is None:                 # fresh runtime: clone the branch
    ROOT = '/content/amoe-lora'
    if not os.path.isdir(os.path.join(ROOT, 'experiments', 'aleph_mnist')):
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                        REPO, ROOT], check=True)

# amoe (src/) and the experiment package (experiments/) go straight on
# sys.path — no build, no editable install to fail on.
for p in (os.path.join(ROOT, 'src'), os.path.join(ROOT, 'experiments')):
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(ROOT)

# deps beyond torch: torchvision (mnist/fashion) + datasets (HF cifar).
# Colab has both; install only what's missing.
for _mod, _pkg in (('torchvision', 'torchvision'), ('datasets', 'datasets')):
    try:
        __import__(_mod)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', _pkg],
                       check=True)

assert os.path.isdir(os.path.join(ROOT, 'experiments', 'aleph_mnist')), \
    f'experiment code not found under {ROOT} — is the {BRANCH} branch pushed?'
print(f'root: {ROOT}\nbranch: {BRANCH}  (in_colab={IN_COLAB})\n'
      f'amoe on path: {os.path.join(ROOT, "src")}')

In [ ]:
# self-heal: if the setup cell above didn't run in THIS kernel (Colab
# restarts drop sys.path; `amoe` can linger from an earlier install and
# mask it), re-add experiments/ so `aleph_mnist` imports regardless.
import sys, os
if not any(p.replace('\\','/').endswith('amoe-lora/experiments') for p in sys.path):
    _R = next((r for r in ('.', '/content/amoe-lora')
               if os.path.isdir(os.path.join(r, 'experiments', 'aleph_mnist'))),
              '/content/amoe-lora')
    if not os.path.isdir(os.path.join(_R, 'experiments', 'aleph_mnist')):
        import subprocess
        subprocess.run(['git','clone','--depth','1','--branch','experimental',
                        'https://github.com/AbstractEyes/amoe-lora', _R], check=True)
    for _p in (os.path.join(_R,'src'), os.path.join(_R,'experiments')):
        if _p not in sys.path: sys.path.insert(0, _p)
    os.chdir(_R)

import torch
import amoe
from amoe import laws
from aleph_mnist import (RunConfig, build_bed, build_trunk, build_heads,
                         pretrain, run, sweep, save_anchor)
from aleph_mnist import probes, plots, vitals
from aleph_mnist.heads import set_adapters

laws.pin_precision()          # fp32, TF32 off — law 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torch {torch.__version__} · device {DEVICE} · amoe {amoe.__version__ if hasattr(amoe, "__version__") else "0.2.2"}')
print(f'binding constant {vitals.BINDING} rad · gate band {vitals.GATE_BAND}'
      f' · escape threshold {laws.BLEND_ESCAPE_RATIO}')

## 1 · The bed

**The starvation is the instrument.** Full MNIST on a 4-block MLP
saturates near 98% and every arm difference disappears into seed noise.
The campaign has been burned by exactly this — a four-arm codebook sweep
on BERT reconstruction found the addressing wasn't load-bearing at all,
because the task was too easy and the model routed around it. The ruling
that came out of it governs this bed: *no champion if it isn't actually
using the alephs.*

So: `d=64`, 4096 class-balanced rows, real headroom left in the task.
Flip `QUICK` off for the full preregistered sweep. Datasets available:
`mnist` · `fashion` (harder) · `cifar10` (hardest). `train_n=None` uses the
**full** set — that plus the d-climb is section 9's workstation workout.

In [ ]:
QUICK = True    # False = the full preregistered sweep (2 seeds, 1500 steps)

CFG = RunConfig(
    d=64, n_blocks=4, tokens=1,
    dataset='mnist', train_n=4096, batch=128,
    steps=400 if QUICK else 1500,
    pretrain_steps=600 if QUICK else 1500,
    lr_head=1e-3, lr_trunk=1e-4,
    probe_every=100 if QUICK else 250,
    log_every=200 if QUICK else 500,
    device=DEVICE,
)
SEEDS = (0,) if QUICK else (0, 1)

bed = build_bed(CFG.dataset, CFG.train_n, seed=CFG.seed).to(DEVICE)
print(f'{bed.name}: train {tuple(bed.xtr.shape)} · test {tuple(bed.xte.shape)}')
print('class balance:', torch.bincount(bed.ytr.cpu()).tolist())
print('neutral sets:', {k: tuple(v.shape) for k, v in bed.neutral.items()})

# the hard-zero law: no input float may be a dead zero
print(f'\nbackground pixel value after normalization: {bed.xtr.min():.4f}'
      f'  (exact zeros in batch: {int((bed.xtr == 0).sum())})')

## 2 · Before training: is the contract intact on a vision trunk?

`amoe.runtime.attach._probe` fingerprints a model by calling
`model(input_ids=...)` with an integer tensor — it is written for causal
LMs. `TinyTrunk` accepts `input_ids` and maps it *deterministically* to a
float input rather than forking `attach()`, which makes the bit-exact
guarantee genuinely testable here.

Two things must hold before any result is worth reading:

1. **Toggle law** — adapters off must equal the base model bit-exactly.
2. **Inertness at attach** — a fresh anchor has a zero-init output head, so
   it should be (near) inert. The campaign documents that the LayerNorm/bias
   path leaks anyway. A 784-in/10-out trunk has no vocabulary to hide that
   leak in, so we can put an exact number on it.

In [ ]:
t = build_trunk(CFG.d, CFG.n_blocks, CFG.tokens, seed=0).to(DEVICE)
before = probes.evaluate(t, bed.xte, bed.yte)
heads, sites, wrappers = build_heads(t, mode='soft', seed=0)

set_adapters(wrappers, False)
off = probes.evaluate(t, bed.xte, bed.yte)
set_adapters(wrappers, True)

print(f'base            ce {before["ce"]:.10f}')
print(f'adapters OFF    ce {off["ce"]:.10f}')
print(f'TOGGLE LAW      {"HOLDS (bit-exact)" if off["ce"] == before["ce"] else "VIOLATED"}')

leak = probes.inertness(t, wrappers, bed.xte[:512])
print(f'\nzero-init leak  max|dlogit| {leak["max_abs_dlogit"]:.6f}'
      f' · mean {leak["mean_abs_dlogit"]:.6f}'
      f' · argmax flips {leak["argmax_flip_frac"]:.4f}')
print('  (law 3 wants ~0; anything above is the documented bias-path leak)')

print('\nparameter census')
print(f'  trunk   {sum(p.numel() for p in t.parameters()) - sum(p.numel() for h in heads for p in h.parameters()):>8,}')
print(f'  adapter {sum(p.numel() for h in heads for p in h.parameters()):>8,}  ({len(heads)} blocks)')

### The arms are parameter-identical — verify, don't assume

A delta between arms is only attributable to the *read* if every other
thing is equal. `none` keeps the codebook parameter so counts match, but
nothing reads it — so it must receive exactly zero gradient, and its
measured drift must be exactly `0.000000`.

In [ ]:
for mode in ('soft', 'sign', 'none'):
    tt = build_trunk(CFG.d, CFG.n_blocks, CFG.tokens, seed=0).to(DEVICE)
    hh, _, ww = build_heads(tt, mode=mode, seed=0)
    n = sum(p.numel() for h in hh for p in h.parameters())
    logits = tt(bed.xtr[:64]).logits
    torch.nn.functional.cross_entropy(logits, bed.ytr[:64]).backward()
    g = vitals.grad_norm_spread({'codebook': [h.addr.codebook for h in hh]})
    print(f'{mode:6} params {n:>8,} · codebook grad-norm {g["norms"]["codebook"]:.3e}')
print('\n`none` must read 0.000e+00 — that is the control being airtight.')

## 3 · Phase 0 + one cell, verbose

**Phase 0 is not optional.** "Frozen" has to mean frozen-*pretrained*, or
the dial's left pole is a random-feature bed and nothing transfers to
exp013's finding. One shared trunk checkpoint per seed; it is also the
baseline curve.

Then one cell at full co-training, watching the vitals move.

In [ ]:
BASE = pretrain(RunConfig(**{**CFG.__dict__, 'seed': 0}), bed)

In [ ]:
import dataclasses
row = run(dataclasses.replace(CFG, mode='soft', trainable_blocks=4), bed, BASE)

print(f'\n{"step":>6} {"ce":>8} {"acc":>7} {"gate":>8} {"drift":>8}'
      f' {"|dh|/|h|":>9} {"g_trunk":>10} {"g_adapt":>10}')
for s in row['traj']:
    dr = sum(s['delta_ratio']) / len(s['delta_ratio'])
    gn = s['grad']['norms']
    print(f'{s["step"]:>6} {s["ce"]:>8.4f} {s["acc"]:>7.4f}'
          f' {s["gate_mean"]:>8.5f} {s["drift_mean"]:>8.5f} {dr:>9.5f}'
          f' {gn.get("trunk", 0):>10.2e} {gn.get("adapter", 0):>10.2e}')

v = row['vitals']
print(f'\ngate mean {v["gate"]["mean"]:.5f} · in 0.012-0.03 band: {v["gate"]["in_band"]}'
      f'   (started at sigmoid(-3) = 0.04743, ABOVE the band)')
print(f'drift mean {v["drift_mean"]:.5f} rad · binding constant {v["binding_target"]}')
print(f'CM CV per block {v["cv"]}  (D=4 sits ~0.99 by design — logged, never gated)')
print(f'\nescape ratios (<= {laws.BLEND_ESCAPE_RATIO} = blend-regime escape): {row["escape"]["ratio"]}')
print(f'sign codes: {row["sign_codes"]["organism_unique"]} unique organism paths'
      f' over {row["sign_codes"]["n_samples"]} samples')
print(f'toggle damage: acc {row["toggle"]["damage_acc"]:+.4f}'
      f' · ce {row["toggle"]["damage_ce"]:+.4f}')

Read the last two lines carefully.

**Toggle damage** is the co-training tax on *detachability*. The toggle law
guarantees the mechanism is bit-exact; it guarantees nothing about what the
trunk has come to depend on. A large positive damage means the artifact is
no longer an adapter — it is half the model, and shipping it as a
detachable anchor would be a lie.

**Escape ratio** is the single-anchor analogue of the P5d blend-escape
gauge. `amoe.diagnostics.diagnose` reads amplitude off the *dispatch*,
which only exists with multiple anchors; with one anchor the honest
equivalent is the residual delta ratio measured on real digits vs
structureless input with identical pixel statistics.

## 4 · The dial

`arms x dial x seeds`, every cell from the same phase-0 checkpoint, one
JSON row per cell appended to `experiments/results/ledger.jsonl`.
The ledger is the evidence; the figures are derived from it and nothing is
ever hand-transcribed.

The `scratch` rows at the end answer the question in its literal form —
adapter snapped onto a *fresh* trunk, both moving from step 0.

In [ ]:
rows = sweep(seeds=SEEDS, base=CFG, bed=bed, bases={0: BASE})
print(f'\n{len(rows)} cells in the ledger')

In [ ]:
print(plots.verdict_table(rows))

## 5 · Figures

Six panels, one question each. The top-left panel is the result: where
`Δ(arm − control)` crosses zero is the boundary.

In [ ]:
%matplotlib inline
fig = plots.figure_set(rows, path='experiments/results/dial.png')

## 6 · Resolving the preregistered forks

Fill this in from the numbers above — **before** reaching for an
explanation of them.

| # | fork | fires if | verdict |
|---|---|---|---|
| 1 | boundary law lands | Δ(soft−none) crosses zero on the dial | |
| 2 | exp012 is scope-limited | no crossing, aleph trails everywhere | |
| 3 | advantage is placement-specific | no crossing, aleph leads everywhere | |
| 4 | sign/soft ordering flips | sign ≥ soft frozen, soft ≥ sign full | |
| 5 | detachability tax | toggle damage grows with the dial | |

Secondary readouts worth recording either way:

- **gate band** — a 7th architecture for the 0.012–0.03 live invariant
  candidate, on a task geometry unlike the other six. There is a standing
  open question asking whether that band is a delta-window prior or a
  property of task geometry; this bed can speak to it.
- **drift** — basin-set-at-init predicts the book barely moves. Does it?
- **sign-code counts** — differentiation must be *structural*;
  gradient-learned alphabets have collapsed 1,594 → 116 unique paths
  before. If co-training collapses the codes here, it is the same disease
  at 1/1000 the cost.

In [ ]:
# quick fork-1 read: the crossing, per dial position
from collections import defaultdict
by = defaultdict(dict)
for r in rows:
    c = r['config']
    if c.get('tag'):
        continue
    by[(c['trainable_blocks'], c['seed'])][c['mode']] = r['final']['ce']
print(f'{"dial":>5} {"soft-none":>12} {"sign-none":>12} {"soft-sign":>12}')
for n in sorted({k[0] for k in by}):
    cells_n = [v for (nn, _), v in by.items() if nn == n]
    def d(a, b):
        xs = [v[a] - v[b] for v in cells_n if a in v and b in v]
        return sum(xs) / len(xs) if xs else float('nan')
    print(f'{n:>5} {d("soft", "none"):>+12.5f} {d("sign", "none"):>+12.5f}'
          f' {d("soft", "sign"):>+12.5f}')
print('\nnegative = the arm beats the control. A SIGN CHANGE down this'
      '\ncolumn is the boundary the campaign never measured.')

## 7 · The artifact round-trips through the public API

A result that can't be shipped isn't finished. This saves the trained
heads as a real `amoe.anchor` v1 checkpoint and puts it through the
runtime verbs on a **fresh** trunk — attach, toggle, detach-with-
verification — with the stock `binding='blocks'` path and no amoe changes.

Note the `address_mode` field in the meta: only `soft` rows are stock
anchors. `sign` and `none` share the checkpoint *layout* but not its
semantics, and plain `attach()` would silently read them as `soft`.

In [ ]:
os.makedirs('experiments/results', exist_ok=True)
path = 'experiments/results/mnist_soft_dial4.anchor.pt'
print('content hash:', save_anchor(row, path))

fresh = build_trunk(CFG.d, CFG.n_blocks, CFG.tokens, seed=0).to(DEVICE)
fresh.load_state_dict({k: v.to(DEVICE) for k, v in BASE.items()})
pre = probes.evaluate(fresh, bed.xte, bed.yte)

h = amoe.attach(fresh, path, binding='blocks')          # ATTACH
post = probes.evaluate(fresh, bed.xte, bed.yte)
print(f'\nattached {h.names} · ce {pre["ce"]:.6f} -> {post["ce"]:.6f}')

with h.all_off():                                        # TOGGLE
    off = probes.evaluate(fresh, bed.xte, bed.yte)
print(f'all_off ce {off["ce"]:.10f} == base {pre["ce"]:.10f} -> {off["ce"] == pre["ce"]}')

h.detach(verify=True)                                    # DETACH
print('detach verified BIT-EXACT against the pre-attach fingerprint')
print(f'post-detach ce {probes.evaluate(fresh, bed.xte, bed.yte)["ce"]:.10f}')

## 8 · What to write back

Whichever fork fired, this bed produces four things worth keeping:

1. **The dial curve** — the boundary law, or a bound on where exp012's
   result stops applying.
2. **A gate-band data point** on a 7th architecture, against a live
   invariant candidate and a standing open question.
3. **The detachability tax** as a function of trunk trainability — which
   decides whether co-trained aleph artifacts can ship as adapters at all.
4. **A portability note for amoe:** the runtime verbs work unmodified on a
   vision trunk via `binding='blocks'`, gated only on the LM-shaped
   `_probe`. A `probe_fn` hook (or a forward-signature sniff) would extend
   the substrate-shim discipline to any architecture with a residual stream.

`experiments/results/ledger.jsonl` is the artifact to keep — every claim
above is re-derivable from it.

## 9 · The substrate climb — run it here, CIFAR included

The seed-0 QUICK pass above said **tie at d=64**: soft never separates
from the passthrough control, and the 4096-row bed let the adapter overfit
(train loss → ~8e-4). Both point the same way — climb the substrate on
**full** training sets until the address bottleneck becomes load-bearing.
exp012's win lived at d=384, so the ladder brackets it.

**Just run the cell below.** `grid()` walks `d × {mnist, fashion, cifar10} ×
seeds`, a fresh trunk per width. **CIFAR loads from Hugging Face**
(`uoft-cs/cifar10`, ~37 s over the CDN) — no slow mirror, no tar to fetch,
nothing to toggle. It prints peak VRAM + s/step at each cell's first step.

**`TRIGRAM = True` (default) is the load-bearing switch.** The single-linear
stem feeds the aleph a *unigram* — one value per position, nothing three-way
to bind — which is why the earlier linear climb tied `soft == none`
everywhere. The byte_emb×3 stem (discovery #16: *channel = n-gram order*;
RGB-as-byte-trigram for cifar, spatial past-only 3-gram for the grayscale
sets) restores the structure the addressed read needs. This is the actual
test of whether the aleph manifests on pixels.

- `CLIMB='quick'` — 3 datasets × 2 widths × 1 seed = 90 cells on full data
  (tens of minutes on the 6000 Pro). The default.
- `CLIMB='full'`  — the whole grid (5 widths × 3 seeds), an overnight run.

Both append to `experiments/results/grid.jsonl`. For a run that survives a
disconnected notebook, the same thing from a terminal is
`python -m aleph_mnist.runner --big`. To point CIFAR at your own HF mirror
instead of `uoft-cs/cifar10`, set `AMOE_CIFAR_HF_REPO` before the cell.

In [ ]:
# The substrate climb. CIFAR comes from HF automatically — no setup.
# CLIMB: 'quick' (~tens of min, real) | 'full' (overnight grid) | 'off'
CLIMB = 'quick'
# TRIGRAM: byte_emb x3 input (discovery #16, channel = n-gram order). The
# single-linear stem feeds the address a UNIGRAM (one value per position,
# nothing 3-way to bind) — that is why the linear-stem climb tied. The
# trigram stem (RGB-as-byte-trigram for cifar; spatial past-only 3-gram for
# mnist/fashion) is what lets the ADDRESSED read manifest (L-AR5).
TRIGRAM = True
MODE = 'trigram' if TRIGRAM else 'linear'

if CLIMB == 'off':
    print("CLIMB='off' — set it to 'quick' or 'full' to run the climb.")
else:
    from aleph_mnist import grid
    # Trigram makes each image a T=784-1024 sequence (~1000x the linear
    # bed's one token), so it uses smaller d and batch to stay tractable;
    # the readout bottleneck (readout_dim) keeps memory flat at any d.
    if CLIMB == 'full':
        if TRIGRAM:
            base = RunConfig(train_n=None, steps=1200, pretrain_steps=1200,
                             batch=256, device=DEVICE, input_mode=MODE)
            dims, seeds = (64, 128, 256, 512), (0, 1)
        else:
            base = RunConfig(train_n=None, steps=2000, pretrain_steps=2000,
                             batch=1024, device=DEVICE, input_mode=MODE)
            dims, seeds = (64, 128, 256, 512, 1024), (0, 1, 2)
    else:  # 'quick' — a real climb that still finishes in a sitting
        if TRIGRAM:
            base = RunConfig(train_n=None, steps=600, pretrain_steps=600,
                             batch=256, device=DEVICE, input_mode=MODE)
            dims, seeds = (64, 128, 256), (0,)
        else:
            base = RunConfig(train_n=None, steps=800, pretrain_steps=800,
                             batch=512, device=DEVICE, input_mode=MODE)
            dims, seeds = (256, 1024), (0,)
    climb = grid(datasets=('mnist', 'fashion', 'cifar10'),  # cifar <- HF
                 dims=dims, seeds=seeds, base=base,
                 root='experiments/data',
                 ledger='experiments/results/grid.jsonl')
    print(f'\nclimb: {len(climb)} cells ({MODE})  ->  results/grid.jsonl')
    print(plots.verdict_table(climb))